# Match Day Agents · Colab fallback

DevFest Trabzon 2026. Trabzonspor host Gaziantep FK on Sunday, and you are the new AI analysis staff.
This notebook runs **the same code as the repo** (`steps/*/agent.py`), one cell per step, for when `adk web` won't run on your laptop.

1. Run the setup cells (clone, install, API key).
2. Run each step. Change the prompt and run it again.
3. To change an agent, open the file in Colab's file browser (left sidebar), edit it, save, and re-run the step's cell: `load()` reloads it.

Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com/apikey).

In [ ]:
# Setup 1/3 · clone the repo (skipped if you're already inside it)
from pathlib import Path

REPO_URL = "https://github.com/REPLACE_ME/match-day-agents"  # TODO: the workshop repo URL

if not Path("pyproject.toml").exists():
    if not Path("match-day-agents").exists():
        !git clone -q {REPO_URL} match-day-agents
    %cd match-day-agents

In [ ]:
# Setup 2/3 · install the repo and its dependencies (google-adk, python-dotenv), about a minute
%pip install -q -e .

In [ ]:
# Setup 3/3 · your Gemini API key (typed, never shown)
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()  # a local .env, if there is one
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API key: ")
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
os.environ.setdefault("MODEL", "gemini-3.8-flash")

In [ ]:
# Helpers: load() imports a step's agent fresh; ask() runs one prompt and prints every tool call and answer.
import importlib
import json
import sys

from google.adk.runners import InMemoryRunner
from google.genai import types


def load(step: str):
    """Re-import steps/<step>/agent.py, so edits to the file take effect."""
    for name in [m for m in sys.modules if m.startswith("steps.")]:
        del sys.modules[name]
    return importlib.import_module(f"steps.{step}.agent").root_agent


async def ask(agent, prompt: str) -> None:
    runner = InMemoryRunner(agent=agent, app_name="colab")
    session = await runner.session_service.create_session(app_name="colab", user_id="staff")
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"YOU: {prompt}\n")
    async for event in runner.run_async(user_id="staff", session_id=session.id, new_message=message):
        for part in (event.content.parts if event.content else []) or []:
            if part.function_call:
                print(f"  [{event.author}] → {part.function_call.name}({json.dumps(part.function_call.args, ensure_ascii=False)})")
            elif part.text and not part.thought:
                print(f"\n[{event.author}]\n{part.text.strip()}\n")
    await runner.close()

## Step 1 · The Scout (one `LlmAgent`)
An agent is an instruction plus tools. The model picks the tools; the tools read the data and do the counting.
Code: `steps/step1_scout/agent.py`, tools: `matchday/tools.py`.

In [ ]:
scout = load("step1_scout")
await ask(scout, "Who's in form for us, and who can't play Sunday?")

## Step 2 · Match Prep (`SequentialAgent`)
Three agents in a fixed order. Each saves its answer with `output_key`; the next reads it through a `{placeholder}`.
Code: `steps/step2_match_prep/agent.py`.

In [ ]:
match_prep = load("step2_match_prep")
await ask(match_prep, "Prepare Sunday's game plan.")

## Step 3 · Scouting Department (`ParallelAgent` inside a `SequentialAgent`)
Three scouts run at the same time; a report writer merges their reports.
Code: `steps/step3_scouting_dept/agent.py`.

In [ ]:
scouting = load("step3_scouting_dept")
await ask(scouting, "Scout Gaziantep for us.")

## Step 4 · Tribün (`LoopAgent`)
The lineup picker proposes, the home crowd objects, until the crowd calls `exit_loop` (or `max_iterations=3` stops it).
Code: `steps/step4_tribun/agent.py`.

In [ ]:
tribun = load("step4_tribun")
await ask(tribun, "Pick the XI for Sunday.")

## Step 5 · Technical Director (orchestrator with `AgentTool`)
The director calls the teams above as tools and picks one by its `description`.
Code: `steps/step5_technical_director/agent.py`. Try the vague descriptions commented out in that file.

In [ ]:
director = load("step5_technical_director")
await ask(director, "Should we rest Salah for Freiburg on Thursday?")

In [ ]:
await ask(director, "If we win 2-0, where do we stand?")

In [ ]:
await ask(director, "Bizi pazara hazırla")

## Bonus · Live data over MCP (presenter only)
The same `build_scout()` with an `McpToolset` instead of local tools. The free server allows 30 requests per minute
per IP, so on the workshop Wi-Fi only the speaker runs this cell.

In [ ]:
live_scout = load("bonus_live_mcp")
await ask(live_scout, "What were Trabzonspor's last three results, and who scored?")